# 00 · Data cleaning

Reads the raw Excel file, removes unusable rows, fixes what can be fixed safely, and writes `data/clean.csv`.

**Rules**
- **Drop a row** if its sales numbers can't be trusted. We don't guess sales figures, and dropping a few store-days costs little.
- **Fix a cell** only when the right value is known. Weather is the same for all three stores on a given date, so a corrupt weather value is copied from the other stores on that date.
- Everything dropped or changed is logged below.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

raw = pd.read_excel("../data/newcastle_sausage_roll_pricing_challenge_20261022.xlsx")
df = raw.copy()
log = []      # one line per rule
dropped = []  # the dropped rows themselves, with the reason
print(f"raw: {len(raw):,} rows")

raw: 2,192 rows


## 1. Drop rows

In [2]:
def drop(rule, mask, why):
    global df
    mask = mask.reindex(df.index, fill_value=False)
    dropped.append(df[mask].assign(reason=rule))
    log.append({"action": "drop row", "rule": rule, "count": int(mask.sum()), "why": why})
    df = df[~mask]

sales = ["total_shop_transactions", "sausage_roll_transactions", "sausage_rolls_sold"]

drop("duplicate row", df.duplicated(),
     "Extra exact copy of a row (same store and date). The first copy is kept, so the day isn't counted twice.")
drop("999999 placeholder", (df.select_dtypes("number") == 999999).any(axis=1),
     "999999 is a placeholder, not a real sales figure (in rolls sold and takings).")
drop("negative count", (df[sales] < 0).any(axis=1),
     "A negative number of transactions or rolls is impossible.")
drop("missing takings", df[["sausage_roll_takings_gbp", "total_shop_takings_gbp"]].isna().any(axis=1),
     "Takings are blank, and revenue is what we compare promotions on.")
drop("rolls sold < roll transactions", df.sausage_rolls_sold < df.sausage_roll_transactions,
     "Every roll transaction sells at least one roll, so this row contradicts itself.")

## 2. Fix weather cells

`temperature_c` and `rain_mm` should match across the three stores on the same date. Where one store has an impossible value,
we use the median of the valid values for that date.

In [3]:
for col, impossible in [("temperature_c", df.temperature_c > 40), ("rain_mm", df.rain_mm < 0)]:
    valid = df[col].mask(impossible)
    df[col] = valid.fillna(valid.groupby(df.date).transform("median"))
    log.append({"action": "fix cell", "rule": f"impossible {col}", "count": int(impossible.sum()),
                "why": "Not physically possible (above 40C, or negative rain). Replaced with the same date's value at the other stores."})

assert df[["temperature_c", "rain_mm"]].notna().all().all()

## 3. What changed

In [4]:
pd.DataFrame(log).style.hide(axis="index").set_properties(**{"white-space": "normal", "text-align": "left"})

action,rule,count,why
drop row,duplicate row,2,"Extra exact copy of a row (same store and date). The first copy is kept, so the day isn't counted twice."
drop row,999999 placeholder,3,"999999 is a placeholder, not a real sales figure (in rolls sold and takings)."
drop row,negative count,3,A negative number of transactions or rolls is impossible.
drop row,missing takings,3,"Takings are blank, and revenue is what we compare promotions on."
drop row,rolls sold < roll transactions,1,"Every roll transaction sells at least one roll, so this row contradicts itself."
fix cell,impossible temperature_c,1,"Not physically possible (above 40C, or negative rain). Replaced with the same date's value at the other stores."
fix cell,impossible rain_mm,3,"Not physically possible (above 40C, or negative rain). Replaced with the same date's value at the other stores."


In [5]:
# the rows that were dropped
show = ["date", "store_id", "sausage_roll_transactions", "sausage_rolls_sold", "sausage_roll_takings_gbp", "total_shop_transactions", "total_shop_takings_gbp"]
pd.concat(dropped)[["reason"] + show]

,reason,date,store_id,sausage_roll_transactions,sausage_rolls_sold,sausage_roll_takings_gbp,total_shop_transactions,total_shop_takings_gbp
14,duplicate row,2024-07-24,NCL-OQ-01,37,43,51.60,256,1271.01
15,duplicate row,2024-07-24,NCL-OQ-01,37,43,51.60,256,1271.01
675,999999 placeholder,2025-03-01,NCL-OQ-01,38,999999,999999.00,184,886.47
1397,999999 placeholder,2025-10-28,NCL-KP-03,60,999999,999999.00,330,1611.28
2112,999999 placeholder,2026-06-23,NCL-OQ-01,67,999999,999999.00,337,1579.25
519,negative count,2025-01-08,NCL-OQ-01,44,62,77.50,-256,1177.33
1203,negative count,2025-08-24,NCL-OQ-01,-20,28,36.40,148,739.12
2015,negative count,2026-05-22,NCL-KP-03,43,-57,76.95,242,1147.12
831,missing takings,2025-04-22,NCL-OQ-01,48,62,NaN,241,NaN
1580,missing takings,2025-12-28,NCL-KP-03,79,99,NaN,366,NaN


In [6]:
print(f"raw {len(raw):,} -> clean {len(df):,} rows ({len(raw) - len(df)} dropped, {(len(raw) - len(df)) / len(raw):.1%})")
print("promotion rows dropped:", int((pd.concat(dropped).promotion != "none").sum()), "| other_promotion rows dropped:", int((pd.concat(dropped).other_promotion != "none").sum()))
print("rows per store:", df.store_id.value_counts().to_dict())

raw 2,192 -> clean 2,180 rows (12 dropped, 0.5%)
promotion rows dropped: 0 | other_promotion rows dropped: 0
rows per store: {'NCL-SJ-02': 730, 'NCL-KP-03': 726, 'NCL-OQ-01': 724}


## 4. Save

In [7]:
df.to_csv("../data/clean.csv", index=False)
print(f"saved data/clean.csv: {df.shape[0]:,} rows x {df.shape[1]} columns")

saved data/clean.csv: 2,180 rows x 17 columns
